### Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings

warnings.filterwarnings('ignore')
print("Library berhasil di-import!")

Library berhasil di-import!


### Dataset

In [2]:
# Memuat file dataset yang sudah terpisah
df_train = pd.read_csv('../dataset/data/train.csv')
df_test = pd.read_csv('../dataset/data/test.csv')

# Menampilkan dimensi data
print(f"Ukuran Data Train : {df_train.shape}")
print(f"Ukuran Data Test  : {df_test.shape}")
print(f"\nKolom : {df_train.columns.tolist()}")
print(f"\nContoh Data Train :")
print(df_train.head())

Ukuran Data Train : (6012038, 6)
Ukuran Data Test  : (1503010, 6)

Kolom : ['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']

Contoh Data Train :
      Timestamp       Open       High        Low      Close     Volume
0  1.553935e+09    4071.17    4071.17    4067.74    4067.74   4.587708
1  1.539865e+09    6431.90    6432.91    6429.81    6432.91   5.358719
2  1.379555e+09     127.96     127.96     127.80     127.80   0.457695
3  1.762841e+09  105322.00  105322.00  105266.00  105267.00   0.339559
4  1.446742e+09     385.77     386.92     385.76     386.92  22.224000


### Pemisahan Target

In [3]:
# Kolom target adalah 'Close' (harga penutupan)
# Kolom fitur adalah semua kolom numerik kecuali 'Close' dan 'Timestamp'

# Kolom yang tidak digunakan sebagai fitur
drop_cols = ['Timestamp', 'Close']

# Pisahkan fitur dan target dari DATA TRAIN
X_train = df_train.drop(columns=drop_cols)
y_train = df_train['Close']

# Pisahkan fitur dan target dari DATA TEST
X_test = df_test.drop(columns=drop_cols)
y_test = df_test['Close']

# Mengelompokkan jenis kolom (semua numerik)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=[object]).columns.tolist()

print("Fitur dan Target berhasil dipisahkan.")
print(f"Kolom Numerik   : {num_cols}")
print(f"Kolom Kategori  : {cat_cols}")

Fitur dan Target berhasil dipisahkan.
Kolom Numerik   : ['Open', 'High', 'Low', 'Volume']
Kolom Kategori  : []


### Preprocessing

**Catatan penting:** Preprocessing (pengisian missing value dan penanganan outlier) hanya menggunakan statistik dari **data train**. Data test tidak diikutsertakan dalam perhitungan statistik untuk menghindari *data leakage*.

In [4]:
# Mengisi Missing Value tanpa melibatkan statistik data test

# 1. Hitung nilai Median kolom numerik dari DATA TRAIN saja
train_medians = X_train[num_cols].median()

# Terapkan pengisian median data train ke X_train saja
X_train[num_cols] = X_train[num_cols].fillna(train_medians)

# Cek missing value pada train
mv_train = X_train.isnull().sum().sum()

# 2. Jika ada kolom kategorikal, hitung modus dari DATA TRAIN saja
if len(cat_cols) > 0:
    train_modes = X_train[cat_cols].mode().iloc[0]
    X_train[cat_cols] = X_train[cat_cols].fillna(train_modes)

print(f"Missing values data train setelah pengisian: {mv_train}")
print("Missing values berhasil ditangani (Data test TIDAK diikutsertakan dalam perhitungan statistik).")

Missing values data train setelah pengisian: 0
Missing values berhasil ditangani (Data test TIDAK diikutsertakan dalam perhitungan statistik).


In [5]:
# Penanganan Outlier menggunakan batas dari data train
# Data test TIDAK diikutsertakan dalam perhitungan statistik outlier

# Simpan batas IQR yang dihitung dari data train
iqr_bounds = {}

for col in num_cols:
    # Hitung Q1, Q3, dan IQR hanya dari X_train
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1

    # Tentukan batas bawah dan batas atas dari DATA TRAIN
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Simpan batas untuk referensi
    iqr_bounds[col] = (lower_bound, upper_bound)

    # Lakukan pembatasan nilai (clipping) hanya pada data train
    X_train[col] = np.clip(X_train[col], lower_bound, upper_bound)
    # Data test TIDAK dilakukan clipping outlier

print("Outlier berhasil ditangani menggunakan batas statistik data train.")
print("Data test TIDAK diikutsertakan dalam penanganan outlier.")

Outlier berhasil ditangani menggunakan batas statistik data train.
Data test TIDAK diikutsertakan dalam penanganan outlier.


### Encoding dan Scaling

Scaler di-*fit* hanya pada **data train**, lalu digunakan untuk men-*transform* data test.

In [6]:
# Transformasi Fitur (Scaling)
# Karena semua kolom numerik, hanya gunakan StandardScaler

scaler = StandardScaler()

# Fit hanya pada data train, lalu transform data train
X_train_scaled = scaler.fit_transform(X_train[num_cols])

# Untuk data test: isi missing value menggunakan statistik dari train
# kemudian transform menggunakan scaler yang sudah di-fit pada train
X_test_filled = X_test[num_cols].fillna(train_medians)
X_test_scaled = scaler.transform(X_test_filled)

print(f"Ukuran X_train setelah transformasi: {X_train_scaled.shape}")
print(f"Ukuran X_test setelah transformasi : {X_test_scaled.shape}")

Ukuran X_train setelah transformasi: (6012038, 4)
Ukuran X_test setelah transformasi : (1503010, 4)


In [7]:
# Reshape data menjadi 3D untuk LSTM
X_train_rnn = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    1,
    X_train_scaled.shape[1]
)

X_test_rnn = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    1,
    X_test_scaled.shape[1]
)

print(X_train_rnn.shape)
print(X_test_rnn.shape)

(6012038, 1, 4)
(1503010, 1, 4)


### Pemodelan LSTM

In [10]:
# Arsitektur Model LSTM
model = Sequential([

    LSTM(
        64,
        activation='tanh',
        recurrent_dropout=0.2,
        input_shape=(
            X_train_rnn.shape[1],
            X_train_rnn.shape[2]
        ),
        return_sequences=False
    ),

    Dropout(0.3),

    Dense(64, activation='relu'),

    Dropout(0.2),

    Dense(32, activation='relu'),

    Dense(1)
])

optimizer = Adam(
    learning_rate=0.0005
)

model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,937 (93.50 KB)

 Trainable params: 23,937 (93.50 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_rnn,
    y_train,

    validation_split=0.2,

    epochs=20,
    batch_size=32,

    callbacks=[early_stop],

    verbose=1
)

Epoch 1/20
150301/150301 ━━━━━━━━━━━━━━━━━━━━ 910s 6ms/step - loss: 31142430.0000 - mae: 2520.9348 - val_loss: 9224590.0000 - val_mae: 1413.2250
Epoch 2/20
 68816/150301 ━━━━━━━━━━━━━━━━━━━━ 7:08 5ms/step - loss: 19719259.7282 - mae: 2339.5152

KeyboardInterrupt: 

### Testing / Evaluasi menggunakan Data Test

Evaluasi model dilakukan menggunakan **data test** yang tidak pernah digunakan selama proses training.

In [ ]:
# Evaluasi menggunakan Data Test

# Melakukan prediksi pada data test
y_pred = model.predict(X_test_rnn).flatten()

# Menghitung berbagai metrik performa regresi
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(" ====== HASIL EVALUASI MODEL PADA DATA TEST ====== ")
print(f"Mean Absolute Error (MAE)      : {mae:.4f}")
print(f"Mean Squared Error (MSE)        : {mse:.4f}")
print(f"Root Mean Squared Error (RMSE) : {rmse:.4f}")
print(f"R-squared Score (R2 Score)     : {r2:.4f}")

### Visualisasi

In [ ]:
# Visualisasi Performa Model

plt.figure(figsize=(12, 5))

# Grafik 1: Grafik Penurunan Loss (MSE)
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Val Loss', color='orange')
plt.title('Grafik Model Loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Grafik 2: Scatter Plot Nilai Aktual vs Prediksi
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred, alpha=0.3, color='purple', s=1)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Perbandingan Nilai Aktual vs Hasil Prediksi')
plt.xlabel('Nilai Aktual (Close)')
plt.ylabel('Nilai Prediksi (Close)')

plt.tight_layout()
plt.show()